# Vim的第一部分：patch embedding

## 每一步的单步调试

In [1]:
from collections import OrderedDict
import torch
from torch import nn

In [2]:
# 模拟一个图片读取结果，图片是1通道28*28，一次读取512张图片成为一个bach
image_files = torch.rand(512, 1, 28, 28)*255

In [3]:
# 1. 经过一次卷积层，进行下采样
patch1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=4, stride=4)(image_files)
patch1.shape

torch.Size([512, 16, 7, 7])

In [4]:
# 2. 把图片拉平 torch.Size([512, 16, 7, 7]) -> torch.Size([512, 16, 49])
patch2 = torch.flatten(patch1, start_dim=2)
patch2.shape

torch.Size([512, 16, 49])

In [5]:
# 3. 改一下通道的位置 torch.Size([512, 16, 49]) -> torch.Size([512, 49, 16])，把通道放到后面去了
patch3 = torch.permute(patch2, [0, 2, 1])
# 4. 添加cls_token，让每张图片自带一个可以用来决策类别的向量
cls_token_one_patch = nn.Parameter(torch.randn(1, 1, 16))
cls_token_all_patch = cls_token_one_patch.expand(512, -1, -1) # torch.Size([512, 1, 16])
# 5. 把专门用来做分类任务的向量通道加到原来的处理过的特征图中 torch.Size([512, 50, 16])
cls_token_all_patch_cls = torch.concat([cls_token_all_patch, patch3], dim=1)
# 6. 加入位置信息编码
pos_embedding = nn.Parameter(torch.randn(1, 50, 16))
patch_cls_pos_embedding = cls_token_all_patch_cls + pos_embedding
cls_token_all_patch.shape

torch.Size([512, 1, 16])

### 关于位置编码的选择
作者认为，使用可学习的参数作为位置编码，虽然有过拟合的风险，但可以提升精确度
使用传统的正余弦那种方式作为位置编码，简单，一致性高。但是有可能影响性能(我理解的就是精确度可能会变低)

## 封装成类

In [10]:
class PatchEmbedding(nn.Module):
    def __init__(self, in_channels, out_channels, patch_size, patch_num, is_dropout):
        super().__init__()
        self.projection = nn.Sequential(
            OrderedDict(
                [
                    ('巧用卷积层把图像切块', nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=patch_size, stride=patch_size)),
                    ('把图片展平，二维变一维', nn.Flatten(2)),
                ]
            ),
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, out_channels), requires_grad=True)
        self.position_embeddings = nn.Parameter(torch.randn(1, patch_num*patch_num + 1, out_channels), requires_grad=True)
        self.dropout = nn.Dropout(p=is_dropout)
    def forward(self, x):
        x = self.projection(x).permute(0, 2, 1)
        # 根据batch_size扩充分类像素点的纬度
        cls_token = self.cls_token.expand(x.shape[0], -1, -1)
        # 把分类像素点加进去
        x = torch.cat((cls_token, x), dim=1)
        # 加入位置信息
        x = self.position_embeddings + x
        x = self.dropout(x)
        return x

In [11]:
input = torch.rand(512,3,28,28)
output = PatchEmbedding(3,16,4,7,True)(input)
print(f'input shape is :{input.shape}')
print(f'output shape is :{output.shape}')

input shape is :torch.Size([512, 3, 28, 28])
output shape is :torch.Size([512, 50, 16])


# Encoder

In [6]:
x = torch.tensor([
    [
        [1, 2],
        [3, 4],
        [5, 6],
        [7, 8]
    ],
    [
        [9, 10],
        [11, 12],
        [13, 14],
        [15, 16]
    ],
    [
        [17, 18],
        [19, 20],
        [21, 22],
        [23, 24]
    ]
])
# print(x.shape)
y = torch.full((x.shape[0],4,1),8)
z = torch.concat((y,x), dim = 2)
z[:,0,0]

tensor([8, 8, 8])